# WLASL encoder training — reset-proof runbook

Free Colab **will** disconnect on you. That is not fixable. What *is* fixable is
how much a disconnect costs, and this notebook is built so the answer is
"almost nothing".

**After any disconnect: reconnect and run every cell from the top again.**
Each one detects what is already done and skips it. Nothing is recomputed twice.

Order matters exactly once: **run cell 5 (the layout check) before cell 6.**
If the layout assumption is wrong, everything trained after it is garbage.

Progress lives in Google Drive, so it survives resets:
- `MyDrive/wlasl/cache/` — preprocessed clips (the slow part, done once)
- `MyDrive/wlasl/sign_encoder_ckpt.pt` — model, saved every ~2 min

## 1. Check the GPU
If this says `cpu`, go **Runtime → Change runtime type → T4 GPU**, then rerun.

In [ ]:
import torch
print("device:", "cuda" if torch.cuda.is_available() else "cpu")
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
else:
    print("!! No GPU. Runtime -> Change runtime type -> T4 GPU, then rerun this cell.")

## 2. Mount Drive
This is what makes progress survive a reset. Approve the popup.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.makedirs('/content/drive/MyDrive/wlasl/cache', exist_ok=True)
print("Drive ready.")

## 3. Get the WLASL landmark data (Kaggle)

**One-time setup:** open the 🔑 **Secrets** panel in the left sidebar and add
your Kaggle credential, with the *Notebook access* toggle ON.

Kaggle changed its token format, so there are two possibilities — use whichever
your account gives you:

| If your token looks like | Add secret named | Value |
|---|---|---|
| one string, `KGAT_...` (current) | `KAGGLE_API_TOKEN` | that whole string |
| a `kaggle.json` with username+key (older) | `KAGGLE_USERNAME` **and** `KAGGLE_KEY` | the two values inside |

Get it at kaggle.com → your avatar → **Settings** → **API** → *Create New Token*.

Put the token in **Secrets**, never in a cell — this notebook lives in a public
GitHub repo, and a token typed into a cell would be published with it.

**Once cell 6 has finished you can skip this cell forever** — training reads only
the Drive cache, never the raw landmarks.

In [ ]:
!pip install -q --upgrade kagglehub

import os
from pathlib import Path
from google.colab import userdata

def _secret(name):
    try:
        return userdata.get(name)
    except Exception:
        return None

tok = _secret("KAGGLE_API_TOKEN")
if tok:
    # Current format. Write the file form too — some client versions only
    # look there, and this costs nothing.
    os.environ["KAGGLE_API_TOKEN"] = tok
    p = Path.home() / ".kaggle" / "access_token"
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(tok); p.chmod(0o600)
    print("auth: KAGGLE_API_TOKEN")
else:
    u, k = _secret("KAGGLE_USERNAME"), _secret("KAGGLE_KEY")
    if not (u and k):
        raise SystemExit(
            "No Kaggle credential found.\n"
            "Open the key icon in the left sidebar and add EITHER\n"
            "  KAGGLE_API_TOKEN  (the KGAT_... string), OR\n"
            "  KAGGLE_USERNAME + KAGGLE_KEY\n"
            "and switch ON 'Notebook access' for each.")
    os.environ["KAGGLE_USERNAME"], os.environ["KAGGLE_KEY"] = u, k
    print("auth: KAGGLE_USERNAME / KAGGLE_KEY")

import kagglehub
data_dir = kagglehub.dataset_download("abd0kamel/mutemotion-output")
print("downloaded to:", data_dir)

# locate the two files we need, wherever the dataset nests them
import glob
npz    = glob.glob(f"{data_dir}/**/landmarks_V3.npz", recursive=True)
parsed = glob.glob(f"{data_dir}/**/WLASL_parsed_data.json", recursive=True)
print("landmarks_V3.npz     ->", npz[0] if npz else "!! NOT FOUND")
print("WLASL_parsed_data.json ->", parsed[0] if parsed else "!! NOT FOUND")
LANDMARKS_NPZ = npz[0] if npz else None
PARSED_JSON   = parsed[0] if parsed else None

## 4. Get the official metadata + your code

`WLASL_v0.3.json` is the only file containing `signer_id` — which signer performed
each clip. Without it there is no signer-disjoint split, and the whole
"learn to ignore who is signing" premise collapses.

Your own repo is cloned here too, so `dtw_common.py` arrives automatically.

In [ ]:
import urllib.request, os, json

# --- official WLASL metadata (has signer_id) ---
CANDIDATES = [
    "https://raw.githubusercontent.com/dxli94/WLASL/master/start_kit/WLASL_v0.3.json",
    "https://raw.githubusercontent.com/dxli94/WLASL/master/data/WLASL_v0.3.json",
]
WLASL_JSON = "/content/WLASL_v0.3.json"
if not os.path.exists(WLASL_JSON):
    for url in CANDIDATES:
        try:
            urllib.request.urlretrieve(url, WLASL_JSON)
            print("got WLASL_v0.3.json from", url)
            break
        except Exception as e:
            print("failed:", url, "->", type(e).__name__)
    else:
        raise SystemExit(
            "Could not fetch WLASL_v0.3.json automatically.\n"
            "Find a copy (many Kaggle WLASL mirrors bundle it), upload it to\n"
            "/content/WLASL_v0.3.json, then rerun this cell.")

with open(WLASL_JSON) as f:
    meta = json.load(f)
n_sign = sum(1 for e in meta for i in e["instances"] if "signer_id" in i)
print(f"{len(meta)} glosses, {n_sign} instances with signer_id")

# --- your project code ---
if not os.path.exists("/content/mk_sign_language"):
    os.system("git clone https://github.com/DianaKostadinova/mk_sign_language.git /content/mk_sign_language")
else:
    os.system("cd /content/mk_sign_language && git pull")
print("repo ready")

## 5. ⚠️ Verify the landmark layout — DO NOT SKIP

`landmarks_V3.npz` gives 553 points per frame with no labels. The code assumes
columns 0–20 are the left hand, 21–41 the right, 42–74 the pose. That assumption
was never fully confirmed, and an earlier guess at the ordering was already
proven wrong once.

If it is wrong, hand features come out as near-constant noise and every epoch you
run trains on garbage — while still producing plausible-looking loss numbers.
Thirty seconds here protects hours later.

**Expect:** clearly non-zero std in x and y for both hands.
**Bad sign:** values at or near `0.0`.

In [ ]:
import os, sys
os.environ["WLASL_JSON"]    = WLASL_JSON
os.environ["PARSED_JSON"]   = PARSED_JSON
os.environ["LANDMARKS_NPZ"] = LANDMARKS_NPZ
os.environ["WLASL_CACHE_DIR"] = "/content/drive/MyDrive/wlasl/cache"
os.environ["WLASL_CKPT"]      = "/content/drive/MyDrive/wlasl/sign_encoder_ckpt.pt"

# 0 = all 2000 glosses. Set to 300 for a quick end-to-end shakedown first.
os.environ["WLASL_MAX_GLOSSES"] = "0"

sys.path.insert(0, "/content/mk_sign_language/alphabet")
sys.path.insert(0, "/content/mk_sign_language/notebooks")
import importlib, encoder_training_wlasl as W
importlib.reload(W)

W.sanity_check_layout()

## 6. Preprocess once → Drive

**This is the cell that used to kill your runs.** Converting ~21k clips into
templates is the slow part, and it used to happen in RAM, scattered randomly
through the first epoch — so a disconnect before that epoch finished threw away
all of it, every time.

Now it runs straight through and writes a shard to Drive every 2000 clips.
A disconnect costs you at most one partial shard. Rerun and it resumes.

Expect roughly 20–45 min the first time. If you get disconnected, just rerun —
the "already cached" count will have gone up.

In [ ]:
W.precompute_templates("train")
W.precompute_templates("test")

## 7. Train

Resumes from the Drive checkpoint automatically. Saves mid-epoch every 2 minutes
and at every epoch boundary, so the most a disconnect can cost is ~2 minutes.

Watch **held-out top-1**, not loss. Loss plateauing near ~1.1 while accuracy keeps
climbing is normal for this contrastive objective — that happened on your AUTSL
run too. Random chance is ~0.05% on 2000 glosses.

Disconnected? Rerun the notebook from the top; this cell picks up where it left off.

In [ ]:
model = W.run_training()

import torch
torch.save(model.state_dict(), "/content/drive/MyDrive/wlasl/sign_encoder_wlasl.pt")
print("saved final encoder to Drive")

## 8. Build the Macedonian gallery

The payoff: because this encoder trains on the same full 21-point hand features
your Macedonian pipeline already uses, your existing `word_templates.npz` works
as-is — no re-extraction.

⚠️ One known mismatch to check before trusting results: your Macedonian
`build_frame` assigns hands to slots by **position in the image** (leftmost
first), while the WLASL side uses MediaPipe's **anatomical** left/right. These
disagree on two-handed signs. Worth resolving here rather than debugging it as
mystery bad accuracy later.

In [ ]:
W.build_macedonian_gallery(
    encoder_path="/content/drive/MyDrive/wlasl/sign_encoder_wlasl.pt",
    templates_npz=f"/content/mk_sign_language/data/landmarks/word_templates.npz",
    out="/content/drive/MyDrive/wlasl/word_embeddings_wlasl.npz",
)